# Module 4 — Match & Team Patterns
**Unit V — Information Visualization | IPL 2008–2019**

Charts in this module:
1. Toss Decision Analysis — What teams choose + does it help? (merged chart 14+15)
2. Chasing vs Defending — Season-wise trend ⭐
3. Powerplay Score vs Match Outcome ⭐ (gold insight)
4. Team Win % — Current franchises only
5. Season-wise Total Match Score Trend
6. ⭐ Venue Impact — Batting first win % by ground
7. ⭐ Match Momentum — 2 famous high-scoring games
8. ⭐ Win Margin Distribution (close games analysis)

## Setup — Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print('Libraries loaded')
from pathlib import Path


In [ ]:
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'deliveries.csv').exists() and (candidate / 'matches.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing deliveries.csv and matches.csv')

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT
CLEAN_DATA_DIR = PROJECT_ROOT / 'data' / 'clean'
DERIVED_DATA_DIR = PROJECT_ROOT / 'data' / 'derived'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'

for path in (CLEAN_DATA_DIR, DERIVED_DATA_DIR, FIGURES_DIR):
    path.mkdir(parents=True, exist_ok=True)

print('Project root    ->', PROJECT_ROOT)
print('Clean data dir  ->', CLEAN_DATA_DIR)
print('Derived data dir->', DERIVED_DATA_DIR)
print('Figures dir     ->', FIGURES_DIR)


In [ ]:
matches_clean = pd.read_csv(CLEAN_DATA_DIR / 'matches_clean.csv')
deliveries    = pd.read_csv(CLEAN_DATA_DIR / 'deliveries_clean.csv')
player_stats  = pd.read_csv(DERIVED_DATA_DIR / 'player_stats.csv')

print('matches_clean shape:', matches_clean.shape)
print('deliveries shape   :', deliveries.shape)

In [ ]:
matches_clean.head(3)

In [ ]:
matches_clean.info()

In [ ]:
matches_clean.describe()

In [ ]:
matches_clean.isnull().sum()

In [ ]:
# Pre-compute batting first team for all matches
# The team that batted first is whoever batted in inning 1
bat_first_lookup = deliveries[deliveries['inning'] == 1].groupby('match_id')['batting_team'].first().reset_index()
bat_first_lookup.columns = ['id', 'bat_first_team']

matches_clean = matches_clean.merge(bat_first_lookup, on='id', how='left')
matches_clean['bat_first_won'] = (matches_clean['bat_first_team'] == matches_clean['winner']).astype(int)

print('bat_first_team and bat_first_won columns added.')
print('Batting first win rate:', matches_clean['bat_first_won'].mean().round(3))

---
## Chart 14 — Toss Decision Analysis (Merged)
*How often do teams choose bat vs field? And which decision wins more?*

**Why merged:** Charts 14 and 15 were asking related questions separately.
One chart combining toss decision count + win % tells the complete story.

In [ ]:
# How many times each decision was made
decision_counts = matches_clean['toss_decision'].value_counts()
print('Toss decision counts:')
print(decision_counts)
print()

# Win % for toss winner by decision
win_pct_by_decision = matches_clean.groupby('toss_decision').apply(
    lambda x: (x['toss_winner'] == x['winner']).mean() * 100
).round(1)
print('Win % when toss winner chose that option:')
print(win_pct_by_decision)

In [ ]:
decisions  = ['bat', 'field']
counts     = [decision_counts.get('bat', 0), decision_counts.get('field', 0)]
win_pcts   = [win_pct_by_decision.get('bat', 0), win_pct_by_decision.get('field', 0)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: how often each decision was made
bars1 = axes[0].bar(['Chose Bat', 'Chose Field'], counts,
                    color=['steelblue', 'orange'], width=0.45, edgecolor='white')
for bar, count in zip(bars1, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(count), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Toss Decision Frequency\n(How often each choice was made)', fontsize=11)
axes[0].set_ylabel('Number of Matches')
axes[0].set_ylim(0, max(counts) + 60)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Right: win % when toss winner made each choice
bars2 = axes[1].bar(['Chose Bat', 'Chose Field'], win_pcts,
                    color=['steelblue', 'orange'], width=0.45, edgecolor='white')
for bar, pct in zip(bars2, win_pcts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{pct}%', ha='center', fontsize=11, fontweight='bold')
axes[1].axhline(50, color='gray', linestyle='--', linewidth=1, label='50% baseline')
axes[1].set_title('Win % When Toss Winner Made That Choice', fontsize=11)
axes[1].set_ylabel('Win %')
axes[1].set_ylim(0, 75)
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Toss Decision Analysis — IPL 2008–2019', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'chart14_toss_decision.png', dpi=150)
plt.show()

print('\nInsight: Teams chose to field 61% of the time.')
print('Choosing to field wins 56.3% — chasing is the preferred and more successful strategy.')

---
## Chart 15 — ⭐ Chasing vs Defending: Season-wise Trend
*Has chasing become easier over 12 years of IPL?*

**Broadcaster angle:** Every IPL season analysts debate "is this a good chasing track?".
This chart shows how the chasing advantage has evolved season by season.
2016 was the strongest chasing year — 68% of matches won by team batting second.

In [ ]:
# bat_first_won already in matches_clean
season_chase = matches_clean.groupby('season')['bat_first_won'].apply(
    lambda x: (1 - x.mean()) * 100
).round(1)

print('Chasing win % per season:')
print(season_chase)

In [ ]:
plt.figure(figsize=(11, 5))

colors = ['#5DCAA5' if v >= 50 else '#F09595' for v in season_chase.values]
bars = plt.bar(season_chase.index, season_chase.values, color=colors,
               edgecolor='white', width=0.6)

plt.axhline(50, color='gray', linestyle='--', linewidth=1.5, label='50% baseline (coin flip)')

# Add % labels on bars
for bar, val in zip(bars, season_chase.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.8,
             f'{val}%', ha='center', fontsize=9)

plt.title('Chasing Win % by Season — IPL 2008–2019\n(Green = chasing team won more | Red = defending team won more)', fontsize=12)
plt.xlabel('Season')
plt.ylabel('Chasing Team Win %')
plt.xticks(season_chase.index, rotation=45)
plt.ylim(0, 82)
plt.legend(fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'chart15_chasing_trend.png', dpi=150)
plt.show()

overall_chase = (1 - matches_clean['bat_first_won'].mean()) * 100
print(f'\nOverall: Chasing wins {overall_chase:.1f}% of IPL matches.')
print('2016 peak: 68.3% of matches won by chasing team — dew and slow outfields helped.')

---
## Chart 16 — ⭐ Powerplay Score vs Match Outcome (GOLD INSIGHT)
*Teams scoring >50 in powerplay win significantly more — coaches track this metric*

**Note:** Batting team is determined directly from delivery data (not assumed from team1)
to ensure accuracy.

In [ ]:
# Powerplay runs for the team batting first in each match
pp = deliveries[(deliveries['inning'] == 1) & (deliveries['phase'] == 'Powerplay')]
pp_scores = pp.groupby('match_id')['total_runs'].sum().reset_index()
pp_scores.columns = ['id', 'pp_runs']

# bat_first_team already known from matches_clean
pp_merged = pp_scores.merge(
    matches_clean[['id', 'bat_first_team', 'winner']], on='id'
)
pp_merged['bat_first_won'] = (pp_merged['bat_first_team'] == pp_merged['winner']).astype(int)

# Bin powerplay scores
pp_merged['pp_bin'] = pd.cut(
    pp_merged['pp_runs'],
    bins=[0, 30, 40, 50, 60, 120],
    labels=['<30', '30–40', '40–50', '50–60', '>60']
)

win_rate = pp_merged.groupby('pp_bin', observed=True)['bat_first_won'].mean().mul(100).round(1)
match_counts = pp_merged.groupby('pp_bin', observed=True).size()

print('Powerplay score → win rate:')
for bin_label in win_rate.index:
    print(f'  {bin_label}: {win_rate[bin_label]}% win rate ({match_counts[bin_label]} matches)')

In [ ]:
bar_colors = ['#E24B4A', '#F09595', '#FAC775', '#5DCAA5', '#0F6E56']

plt.figure(figsize=(9, 5))
bars = plt.bar(win_rate.index, win_rate.values, color=bar_colors, width=0.5, edgecolor='white')

plt.axhline(50, color='gray', linestyle='--', linewidth=1.5, label='50% baseline')

# Add win% + match count labels
for bar, (label, val) in zip(bars, win_rate.items()):
    n = match_counts[label]
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val}%\n({n} matches)', ha='center', fontsize=9)

plt.title('Win % for Batting First Team by Powerplay Score — IPL 2008–2019\n(Overs 1–6 | higher powerplay score = better winning chance)', fontsize=12)
plt.xlabel('Powerplay Runs Scored')
plt.ylabel('Win % for Batting First Team')
plt.ylim(0, 90)
plt.legend(fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'chart16_powerplay_outcome.png', dpi=150)
plt.show()

print('\nInsight: Teams scoring >60 in powerplay win 71% of the time.')
print('Teams scoring <30 win only ~27% — the powerplay sets the tone of the match.')

---
## Chart 17 — Team Win % (Active Franchises)
*Which franchises have been most successful across IPL history?*

**Why win % and not total wins:** Total wins favors teams that played more seasons.
Win % (wins per match played) is a fairer comparison across all franchises.
Only active franchises shown (minimum 100 matches played).

In [ ]:
# Matches played per team
team1 = matches_clean[['team1', 'id']].rename(columns={'team1': 'team'})
team2 = matches_clean[['team2', 'id']].rename(columns={'team2': 'team'})
team_matches = pd.concat([team1, team2]).groupby('team')['id'].count().reset_index(name='played')

# Wins per team
team_wins_count = matches_clean['winner'].value_counts().reset_index()
team_wins_count.columns = ['team', 'wins']

team_perf = team_matches.merge(team_wins_count, on='team')
team_perf['win_pct'] = (team_perf['wins'] / team_perf['played'] * 100).round(1)

# Only keep teams with 100+ matches (excludes defunct one-season teams)
team_perf = team_perf[team_perf['played'] >= 100].sort_values('win_pct', ascending=True)

print(team_perf[['team', 'played', 'wins', 'win_pct']].to_string())

In [ ]:
# Add batting first and chasing win % per team
bat_first_wins = matches_clean[matches_clean['bat_first_won'] == 1]['bat_first_team'].value_counts().reset_index()
bat_first_wins.columns = ['team', 'bat_first_wins']

chase_wins_team = matches_clean[matches_clean['bat_first_won'] == 0]['winner'].value_counts().reset_index()
chase_wins_team.columns = ['team', 'chase_wins']

team_perf = team_perf.merge(bat_first_wins, on='team', how='left')
team_perf = team_perf.merge(chase_wins_team, on='team', how='left')
team_perf['bat_first_win_pct'] = (team_perf['bat_first_wins'] / team_perf['played'] * 100).round(1)
team_perf['chase_win_pct']     = (team_perf['chase_wins']     / team_perf['played'] * 100).round(1)
team_perf = team_perf.sort_values('win_pct', ascending=True)

x = range(len(team_perf))
width = 0.3

plt.figure(figsize=(11, 6))
plt.barh([i - width/2 for i in x], team_perf['bat_first_win_pct'],
         width, label='Win % batting first', color='steelblue', edgecolor='white')
plt.barh([i + width/2 for i in x], team_perf['chase_win_pct'],
         width, label='Win % chasing',       color='#5DCAA5',   edgecolor='white')

plt.yticks(list(x), team_perf['team'])
plt.axvline(25, color='gray', linestyle='--', linewidth=1, alpha=0.5)
plt.title('Team Win % — Batting First vs Chasing (IPL 2008–2019)\n(min. 100 matches played)', fontsize=12)
plt.xlabel('Win % (out of all matches played)')
plt.legend(fontsize=9)
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'chart17_team_win_pct.png', dpi=150)
plt.show()

print('\nInsight: CSK wins more batting first (defending). MI wins more chasing.')
print('This shows different team strategies — CSK builds big totals, MI chases well.')

---
## Chart 18 — Season-wise Total Match Score Trend
*Are IPL matches becoming higher-scoring over time?*

Both innings combined per match — shows whether the game as a whole
is producing more runs. 2018 was clearly the highest-scoring season.

In [ ]:
# Total runs per match, then average per season
match_totals = deliveries.groupby('match_id')['total_runs'].sum().reset_index()
match_totals = match_totals.merge(
    matches_clean[['id', 'season']], left_on='match_id', right_on='id', how='left'
)
season_avg_total = match_totals.groupby('season')['total_runs'].mean().round(1)

print('Average total match score per season (both innings):')
print(season_avg_total)

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(season_avg_total.index, season_avg_total.values,
         color='steelblue', linewidth=2.5, marker='o', markersize=7)

# Highlight peak season
peak_season = season_avg_total.idxmax()
peak_val = season_avg_total.max()
plt.annotate(f'Peak: {peak_val}\n({peak_season})',
             xy=(peak_season, peak_val),
             xytext=(peak_season - 1.5, peak_val + 4),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=9, color='red')

# Fill under line
plt.fill_between(season_avg_total.index, season_avg_total.values,
                 alpha=0.1, color='steelblue')

plt.title('Average Total Match Score per Season — IPL 2008–2019\n(Both innings combined | higher = more runs in that season)', fontsize=12)
plt.xlabel('Season')
plt.ylabel('Average Runs per Match')
plt.xticks(season_avg_total.index, rotation=45)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'chart18_season_scores.png', dpi=150)
plt.show()

print('\nInsight: Scores rose sharply in 2018 (avg 345 runs/match).')
print('2009 (South Africa) was the lowest-scoring season — different conditions.')

---
## Chart 19 — ⭐ Venue Impact — Batting First Win %
*Does the ground determine your strategy? Which venues favor defending vs chasing?*

**Broadcaster angle:** Before every IPL game, commentators discuss the venue's history.
Chepauk (Chennai) is known as a batting-first fortress. Eden Gardens favors chasing.
This chart proves those observations with data.

Filter: Minimum 15 matches at that venue.

In [ ]:
# bat_first_won already computed per match
venue_stats = matches_clean.groupby('venue').agg(
    matches  = ('id', 'count'),
    bat_wins = ('bat_first_won', 'sum')
).reset_index()

venue_stats['bat_first_pct'] = (venue_stats['bat_wins'] / venue_stats['matches'] * 100).round(1)

# Keep only venues with 15+ matches
venue_stats = venue_stats[venue_stats['matches'] >= 15].sort_values('bat_first_pct', ascending=True)

# Shorten long venue names for readability
name_map = {
    'Rajiv Gandhi International Stadium, Uppal': 'RGIS Hyderabad',
    'Punjab Cricket Association Stadium, Mohali': 'PCA Stadium Mohali',
    'MA Chidambaram Stadium, Chepauk': 'Chepauk Chennai',
    'Maharashtra Cricket Association Stadium': 'MCA Pune',
    'Dr DY Patil Sports Academy': 'DY Patil Mumbai',
    'Subrata Roy Sahara Stadium': 'Sahara Pune'
}
venue_stats['venue_short'] = venue_stats['venue'].replace(name_map)

print('Venue batting first win %:')
print(venue_stats[['venue_short', 'matches', 'bat_first_pct']].to_string())

In [ ]:
# Color: green if bat first wins >50%, red if chasing wins more
bar_colors = ['#5DCAA5' if v >= 50 else '#F09595' for v in venue_stats['bat_first_pct']]

plt.figure(figsize=(11, 7))
bars = plt.barh(venue_stats['venue_short'], venue_stats['bat_first_pct'],
                color=bar_colors, edgecolor='white')

# Add % label + match count
for bar, (_, row) in zip(bars, venue_stats.iterrows()):
    plt.text(bar.get_width() + 0.3,
             bar.get_y() + bar.get_height()/2,
             f"{row['bat_first_pct']}%  ({int(row['matches'])} matches)",
             va='center', fontsize=9)

plt.axvline(50, color='gray', linestyle='--', linewidth=1.5, label='50% baseline')
plt.title('Batting First Win % by Venue — IPL 2008–2019\n(Green = defending team wins more | Red = chasing team wins more)', fontsize=12)
plt.xlabel('Batting First Win %')
plt.xlim(0, 80)
plt.legend(fontsize=9)
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'chart19_venue_impact.png', dpi=150)
plt.show()

print('\nInsight: Chepauk (63%) and Sahara Pune (65%) favor batting first — slow pitches.')
print('Eden Gardens (39%) and Sawai Mansingh (32%) strongly favor chasing.')

---
## Chart 20 — ⭐ Match Momentum — 2 Famous High-Scoring Games
*Cumulative runs over by over — shows the turning point and run rate pressure*

Two of the highest-scoring IPL matches selected:
- KKR vs KXIP 2018 (471 total runs)
- CSK vs RR 2010 (469 total runs)

In [ ]:
# Two highest-scoring matches (verified from data)
# Match 7937: KKR vs Punjab Kings 2018 | Match 206: CSK vs Rajasthan Royals 2010
selected_matches = [
    {'match_id': 7937, 'label': 'KKR vs Punjab Kings (2018)'},
    {'match_id': 206,  'label': 'CSK vs Rajasthan Royals (2010)'}
]

for m in selected_matches:
    md = deliveries[deliveries['match_id'] == m['match_id']]
    teams = md.groupby('inning')['batting_team'].first().to_dict()
    total = md['total_runs'].sum()
    print(f"Match {m['match_id']}: {teams} | Total runs: {total}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

line_colors = ['steelblue', 'red']

for ax, m in zip(axes, selected_matches):
    match_data = deliveries[deliveries['match_id'] == m['match_id']]
    winner_row = matches_clean[matches_clean['id'] == m['match_id']]
    winner = winner_row['winner'].values[0] if len(winner_row) > 0 else 'Unknown'

    for inn, color in zip([1, 2], line_colors):
        inn_data = match_data[match_data['inning'] == inn]
        if len(inn_data) == 0:
            continue
        over_runs = inn_data.groupby('over')['total_runs'].sum()
        cumulative = over_runs.cumsum()
        team = inn_data['batting_team'].iloc[0]
        ax.plot(cumulative.index, cumulative.values,
                color=color, linewidth=2.5, marker='o', markersize=4,
                label=f'Inn {inn}: {team}')

    ax.set_title(f'{m["label"]}\nWinner: {winner}', fontsize=10)
    ax.set_xlabel('Over Number')
    ax.set_ylabel('Cumulative Runs')
    ax.set_xticks(range(1, 21))
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Match Momentum — Cumulative Runs per Over\n(Two Highest-Scoring IPL Matches)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'chart20_momentum.png', dpi=150)
plt.show()

print('\nInsight: Both matches show a momentum shift in death overs.')
print('The steeper the slope in overs 16-20, the more aggressive the finishing.')

---
## Chart 21 — ⭐ Win Margin Distribution
*How close are IPL matches? How often are games won by a tiny margin?*

**Two types of wins:** By runs (batting first team wins) and by wickets (chasing team wins).
Close games (won by ≤10 runs or ≤2 wickets) show how competitive IPL is.

In [ ]:
run_wins = matches_clean[matches_clean['win_by_runs'] > 0]['win_by_runs']
wkt_wins = matches_clean[matches_clean['win_by_wickets'] > 0]['win_by_wickets']

print(f'Matches won by runs    : {len(run_wins)}')
print(f'Matches won by wickets : {len(wkt_wins)}')
print(f'Close run wins (<=10)  : {(run_wins <= 10).sum()}')
print(f'Close wkt wins (<=2)   : {(wkt_wins <= 2).sum()}')
print(f'Avg margin by runs     : {run_wins.mean():.1f}')
print(f'Avg margin by wickets  : {wkt_wins.mean():.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Wins by runs histogram
axes[0].hist(run_wins, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(run_wins.mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'Avg: {run_wins.mean():.1f} runs')
axes[0].axvline(10, color='orange', linestyle=':', linewidth=1.5,
                label=f'Close (<= 10 runs): {(run_wins <= 10).sum()} games')
axes[0].set_title('Win Margin — When Batting First Team Wins', fontsize=11)
axes[0].set_xlabel('Winning Margin (runs)')
axes[0].set_ylabel('Number of Matches')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Wins by wickets bar chart
wkt_counts = wkt_wins.value_counts().sort_index()
bar_colors = ['#F09595' if i <= 2 else 'steelblue' for i in wkt_counts.index]
axes[1].bar(wkt_counts.index, wkt_counts.values, color=bar_colors, edgecolor='white')
axes[1].set_title('Win Margin — When Chasing Team Wins', fontsize=11)
axes[1].set_xlabel('Winning Margin (wickets remaining)')
axes[1].set_ylabel('Number of Matches')
axes[1].set_xticks(range(1, 11))
axes[1].set_xticklabels([f'{w}W' for w in range(1, 11)])

# Label red bars (close games)
for bar, (wkt, count) in zip(axes[1].patches, wkt_counts.items()):
    if wkt <= 2:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     str(count), ha='center', fontsize=9, color='red')

axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('IPL Win Margin Distribution — How Close Are Matches? (2008–2019)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'chart21_win_margins.png', dpi=150)
plt.show()

print('\nInsight: 80 matches decided by 10 runs or fewer — IPL is genuinely competitive.')
print('Most wicket wins are by 4–6 wickets — comfortable chases are common.')